In [3]:
blob_account_name = "azureopendatastorage"
blob_container_name = "nyctlc"
blob_relative_path = "yellow"

wasbs_path = f'wasbs://{blob_container_name}@{blob_account_name}.blob.core.windows.net/{blob_relative_path}'
print(wasbs_path)

blob_df = spark.read.parquet(wasbs_path)

StatementMeta(, 65171838-5f58-4c0e-b248-60896cf8a787, 5, Finished, Available, Finished, False)

wasbs://nyctlc@azureopendatastorage.blob.core.windows.net/yellow


In [4]:
file_name = "yellow_taxi"

output_parquet_path = f"abfss://DataEngineering@onelake.dfs.fabric.microsoft.com/Data.Lakehouse/Files/RawData/{file_name}"
print(output_parquet_path)

blob_df.limit(100).write.mode("overwrite").parquet(output_parquet_path)



StatementMeta(, 65171838-5f58-4c0e-b248-60896cf8a787, 6, Finished, Available, Finished, False)

abfss://DataEngineering@onelake.dfs.fabric.microsoft.com/Data.Lakehouse/Files/RawData/yellow_taxi


In [6]:
from pyspark.sql.functions import col, to_timestamp, current_timestamp, year, month

raw_df = spark.read.parquet(output_parquet_path)

filtered_df = raw_df.withColumn("dataload_datetime", current_timestamp())

filtered_df = filtered_df.filter(col("storeAndFwdFlag").isNotNull())

table_name = "yellow_taxi"
# Load to delta table
filtered_df.write.format("delta").mode("append").saveAsTable(table_name)

display(filtered_df.limit(1))

StatementMeta(, 65171838-5f58-4c0e-b248-60896cf8a787, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5965f059-8ff7-4e70-a418-b52d5e632829)

In [8]:
## query delta table using sql 

delta_table_name = "yellow_taxi"
table_df = spark.read.format("delta").table(delta_table_name)

table_df.createOrReplaceTempView("yellow_taxi_temp")

table_df = spark.sql('SELECT * FROM yellow_taxi_temp')

display(table_df.limit(10))


StatementMeta(, 65171838-5f58-4c0e-b248-60896cf8a787, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6417702a-fd17-4a25-a3fb-4f90e1f41c8a)